In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import xarray as xr
from scipy.ndimage import uniform_filter1d
from copy import copy

experimentFolder = r'O:\HybridDune experiment\data RBR, OSSI\netcdf'                   # path where the data is stored, change as needed


In [4]:
def movmean(x, N):
    # calculate moving mean. NB: this divides values at the edges by the window length, instead of the available number of values
    y = uniform_filter1d(x, size=N, mode='constant') # for even window: backward avg. So window 2: x_m(i)=[x(i-1)+x(i)]/2. x_m(i=1) = x(i=1)/2

    # compensate edges for number of values, i.e. the truncated window length
    S1 = np.arange(np.ceil(N/2), N)
    S2 = np.ones(len(x)-N+1)*N
    S3 = np.arange(N-1, np.floor(N/2), -1)
    S = np.concatenate((S1, S2, S3)) 
    return y * N / S


In [5]:

instrumentname_all = ['ref P_BS1 (RBR)', 'S1 P_BS3 (RBR)', 'S2 P_BS3 (RBR)', 'S3 P_BS3 (RBR)', 'S4 P_BS3 (RBR)','S1 P_BS2 (RBR)'] 
#                      RBR4(ref),         RBR5,             RBR1,             RBR6,             RBR2,            RBR3
sf_all            = [       8,              16,               8,              16,                 8,              8]              # [hz] sampling frequency
offset_all        = [    -543,            -147,             268,              38,               320,             64]              # [Pa] offset to be added to the pressure data, for instrument calibration

In [6]:
# # CALCULATE ATMOSPHERIC AIR PRESSURE, ...
# Set smoothing window for atmpospheric pressure
t_smooth_air = 10 # [s]     # measured with 8 hz. But p_water and p_air are measured up to 100m apart (and p_air inside). Affected different by wind gusts, so filter out short-term variation

# Open raw data file  of reference sensor -------------------------------------------------------------------
dataFile = os.path.join(experimentFolder,'raw NetCDF', 'Deployment period 1', 'Pressure sensor ' + instrumentname_all[0]   + ' raw data - period 1.nc')
with xr.open_dataset(dataFile) as ds:
    # Calibrate referense sensor
    ds['p'] = ds['p'] + offset_all[0]  # add offset to pressure data, for instrument calibration

    # crop dataset to the time range of interest. 5 RBRs with about the same end time, make exactly the same
    t0 = datetime(2024, 12, 17, 11, 0, 0)         # first full hour that all 5 instruments were installed at the beach
    t_end = datetime(2024, 12, 23, 13, 0)         # last full hour that all 5 instruments were installed at the beach

    ds_air_8hz = ds.sel(t=slice(t0, t_end))

# Determine moving average. 
tAir_8hz = ds_air_8hz['t']
pAir_smooth_8hz = movmean(ds_air_8hz['p'], 8*t_smooth_air) # smooth over 8hz * n seconds

# interpolate for 16hz sensors
tAir_16hz   = pd.date_range(t0, t_end, freq='{}s'.format(1 / 16)) # 16hz time vector
pAir_smooth_16hz = np.interp(tAir_16hz, tAir_8hz, pAir_smooth_8hz) # interpolate to 16hz time vector

# Add to dataset
ds_air_8hz['pAir_smooth'] = (('t'), pAir_smooth_8hz )
ds_air_16hz = xr.Dataset( # make dataset for 16hz air pressure
    data_vars={'pAir_smooth': (('t',), pAir_smooth_16hz)},
    coords={'t': tAir_16hz} )

In [ ]:
# Instrument to be corrected, filtered
for i in range(0,6): #[0, 1,2,3,4,5]:
    instrumentName = instrumentname_all[i]                                                                               # designated name of the instrument
    dataFile = os.path.join(experimentFolder,'raw NetCDF', 'Deployment period 1', 'Pressure sensor ' + instrumentname_all[i]   + ' raw data - period 1.nc')

    with xr.open_dataset(dataFile) as ds0:
        ds0 = ds0.sel(t=slice(t0, t_end))  # crop dataset to the time range of interest: when all instruments were at the beach
        ds0 = ds0.rename({'p': 'p_abs'})   # rename p to p_abs

    # Relative pressure: correct the pressure signal with pAir
    if i == 0: # refP1 RBR4
        ds0['p_air'] = ds_air_8hz['pAir_smooth'] # air pressure, already calibrated
    elif sf_all[i] == 8:    
        ds0['p_rel'] = ds0['p_abs'] + offset_all[i] - ds_air_8hz['pAir_smooth']   # relative pressure. Add calibration offset per instrument. ds_air already calibrated 
    else:
        ds0['p_rel'] = ds0['p_abs'] + offset_all[i] - ds_air_16hz['pAir_smooth'] 

    # Correct negative pressures
    if i != 0: # only for non-reference sensors
        block_mask = ds0['p_rel'] < 0
        ds0['p_rel'] = ds0['p_rel'].where(~block_mask, 0)               # set negative pressures to zero
        
    # Add metadata
    cal_text = '{}'.format(offset_all[i]) + ' Pa added to raw pressure, based on the period 23dec, 19:11 to 19:26, during the calibration test'
    ds0.p_abs.attrs = {'units': 'Pa', 'long_name': 'Absolute pressure', 'comments': 'calibrated', 'calibration': cal_text}

    if i == 0: # refP1 RBR4
        cal_text = '{}'.format(offset_all[0]) + ' Pa added to raw pressure, based on the period 23dec, 19:11 to 19:26, during the calibration test'
        ds0['p_air'].attrs = {'units': 'Pa', 'long_name': 'Air pressure', 'comments': '10s moving average and calibrated','calibration': cal_text}
        ds0.attrs['summary'] = 'Hybrid-Dune campaign: air pressure'
    else:
        cal_text = '{}'.format(offset_all[i]) + ' Pa added to raw pressure of instrument (and ' + '{}'.format(offset_all[0]) + ' Pa added to air pressure of sensor refP1), based on the period 23dec, 19:11 to 19:26, during the calibration test'
        ds0['p_rel'].attrs = {'units': 'Pa', 'long_name': 'Relative pressure', 'comments': 'corrected for air pressure and calibrated','calibration': cal_text}
        ds0.attrs['summary'] = 'Hybrid-Dune campaign: pressure corrected for air pressure'

    ds0.attrs['start time'] = pd.to_datetime(ds.t.values[0]).strftime("%d-%b-%Y %H:%M:%S")
    ds0.attrs['end time'] =   pd.to_datetime(ds.t.values[-1]).strftime("%d-%b-%Y %H:%M:%S")

    # Save to netcdf -------------------------------
    ds0.p_abs.values = np.round(ds0.p_abs.values)    # Round pressure to 1 Pa = 0.1 mm 
    if i == 0:    # refP1 RBR4
        ds0.p_air.values = np.round(ds0.p_air.values)    
        ncFilePath = os.path.join(experimentFolder, 'QC', 'Deployment period 1', 'Pressure sensor ' + instrumentName + ' QC p_air - period 1.nc')
    else:
        ds0.p_rel.values = np.round(ds0.p_rel.values)   
        ncFilePath = os.path.join(experimentFolder, 'QC', 'Deployment period 1', 'Pressure sensor ' + instrumentName + ' QC p_rel - period 1.nc')
    encoding = {var: {"zlib": True, "complevel": 4} for var in list(ds0.data_vars) + list(ds0.coords)}  # Apply deflate compression to all variables and coordinates in netCDF
    ds0.encoding = encoding  # add the encoding to the dataset (not really necessary, but allows retrieval later on)

    if not os.path.isdir(os.path.join(experimentFolder,'QC','Deployment period 1')):
        os.mkdir(os.path.join(experimentFolder,'QC','Deployment period 1'))
    ds0.to_netcdf(ncFilePath, encoding=encoding) # save to netcdf